![Week 1 Day 1](../../assets/banners/w1d1.png)

# Week 1 Day 1 - Solution: The Stricter Marker

The Day 1 exercise asked you to make the marker **stricter**: the model should give the definition marks only if Sam says *processes* (not "programs"), and must explain any mark it withholds. Then check whether that makes it more consistent, and try a weaker answer.

> 💡 Try the exercise yourself first (it's at the end of `week1/day1.ipynb`). Struggling a little is where the learning happens! Then come back and compare.

In [ ]:
# Imports and environment first, all in one place

import os
import re
from collections import Counter

from dotenv import load_dotenv
from IPython.display import Markdown, display
from langchain.chat_models import init_chat_model

from gradewise.config import settings

load_dotenv(override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")

if openai_api_key:
    print(f"✅ OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("❌ OpenAI API Key not set. Please see Part 5 of the setup guide (and remember to SAVE .env!)")
    print("   Using another provider? That's fine: see setup/SWAP_PROVIDER.md")

print(f"🧠 Everyday model from your .env: {settings.chat_model}")

## Part 1: The same question, scheme and answer as Day 1

In [ ]:
question = "Q3. What is a deadlock? Name TWO of the four necessary conditions for a deadlock. (4 marks)"

marking_scheme = """
- 2 marks: a correct definition (a set of processes, each waiting for a resource held by another, so none can proceed)
- 1 mark each (max 2): any two of mutual exclusion, hold and wait, no preemption, circular wait
"""

sam_answer = """
A deadlock is when two programs are stuck because each one is waiting for the other one to finish.
Conditions: mutual exclusion and circular wait.
"""

## Part 2: A stricter prompt

Three changes make the difference:
1. **Explicit rules** instead of "use the marking scheme": the word *processes* is required, and the resource idea must be there.
2. **Show your working:** a line per mark point, so every withheld mark is explained.
3. **A fixed last line** (`MARK: n/4`) that code can read with a regular expression. (Day 3 does this properly with structured output.)

In [ ]:
STRICT_PROMPT = """You are a strict but fair exam marker. Mark the student's answer using ONLY the marking scheme
and these extra rules:
- Award the 2 definition marks ONLY if the answer says "processes" (or "threads"). "Programs" is not enough.
- The definition must also say each one waits for a RESOURCE held by another. Without that, award at most 1.
- Award 1 mark for each correct condition, at most 2.
- For EVERY mark you withhold, say which rule caused it.

Question: {question}
Marking scheme: {scheme}
Student answer: {answer}

Reply with one line per mark point, then a last line in exactly this form: MARK: <number>/4"""


def mark(llm, answer: str) -> tuple[float | None, str]:
    """Ask for a mark and read it from the MARK: line. Returns (mark or None, full reply)."""
    reply = llm.invoke(STRICT_PROMPT.format(question=question, scheme=marking_scheme, answer=answer)).content
    found = re.search(r"MARK:\s*([\d.]+)\s*/\s*4", reply)
    return (float(found.group(1)) if found else None), reply


print(STRICT_PROMPT.format(question=question, scheme=marking_scheme, answer=sam_answer))

In [ ]:
llm = init_chat_model(settings.chat_model)

sam_mark, reply = mark(llm, sam_answer)
display(Markdown(reply))

## Part 3: Is it more consistent?

Run the same marking five times with `.batch()` and count the marks. With the strict rules, you should see the same mark (usually **2/4**: the conditions earn 2, the definition loses both marks for "programs" and the missing resource idea) far more often than with Day 1's loose prompt.

In [ ]:
prompt = STRICT_PROMPT.format(question=question, scheme=marking_scheme, answer=sam_answer)
replies = llm.batch([prompt] * 5)

marks = [re.search(r"MARK:\s*([\d.]+)", r.content) for r in replies]
print("Marks over 5 runs:", Counter(m.group(1) if m else "unreadable" for m in marks))

## Bonus: a weaker answer

A good marking prompt should also get the *low* end right. This answer confuses deadlock with starvation and names one wrong condition.

In [ ]:
weak_answer = """
Deadlock is when a process never gets the CPU because other processes keep going first.
Conditions: starvation and mutual exclusion.
"""

weak_mark, reply = mark(llm, weak_answer)
display(Markdown(reply))
print(f"Sam: {sam_mark}/4 · weak answer: {weak_mark}/4  (expect the weak one to score 1: only mutual exclusion)")

## What we learned

- Specific rules beat "use the scheme": the model can't apply rules it was never told.
- Asking for working makes marks **explainable**, and usually more consistent.
- A fixed output line is a fragile way to read a mark. On **Day 3** you'll replace it with `with_structured_output`, and in **week 5** GradeWise marks per criterion, with evidence.